# Audio Feature Exploration

Visualize the MFCC + Mel-spectrogram features used by the SoundSentinel classifier.

**Prerequisites:**
- Run `python training/download_esc50.py` first to get the dataset.
- Install: `pip install librosa matplotlib numpy`

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

AUDIO_ROOT = Path('../../data/ESC-50-master/audio')
print(f'Looking in: {AUDIO_ROOT.resolve()}')
wavs = list(AUDIO_ROOT.glob('*.wav'))[:5]
print(f'Found {len(wavs)} sample wavs to inspect')

## 1. Waveform

In [ ]:
y, sr = librosa.load(str(wavs[0]), sr=22050, duration=5.0)
print(f'Sample rate: {sr}, duration: {len(y)/sr:.2f}s, samples: {len(y)}')

plt.figure(figsize=(12, 3))
librosa.display.waveshow(y, sr=sr)
plt.title(f'Waveform — {wavs[0].name}')
plt.xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 2. Mel-Spectrogram

In [ ]:
mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(12, 4))
librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', fmax=8000)
plt.colorbar(format='%+2.0f dB')
plt.title('Mel-Spectrogram')
plt.tight_layout()
plt.show()

## 3. MFCC (Mel-Frequency Cepstral Coefficients)

The first 40 MFCCs are what the SVM/XGBoost classifier actually receives.

In [ ]:
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
print(f'MFCC shape: {mfcc.shape} (40 coefficients x {mfcc.shape[1]} frames)')

plt.figure(figsize=(12, 5))
librosa.display.specshow(mfcc, sr=sr, x_axis='time')
plt.colorbar()
plt.title('MFCC')
plt.ylabel('MFCC coefficient')
plt.tight_layout()
plt.show()

## 4. Feature Vector (mean + std)

predict.py condenses each MFCC matrix into an 80-dim vector by stacking mean and std.

In [ ]:
features = np.hstack([mfcc.mean(axis=1), mfcc.std(axis=1)])
print(f'Feature shape: {features.shape}')  # (80,)

plt.figure(figsize=(12, 3))
plt.bar(range(40), features[:40], color='steelblue', label='mean')
plt.bar(range(40), features[40:], color='salmon', alpha=0.7, label='std')
plt.xlabel('MFCC coefficient index')
plt.ylabel('Value')
plt.title('80-dim feature vector (mean + std)')
plt.legend()
plt.tight_layout()
plt.show()